# Prompt Engineering — Iterative Improvement Process

---

## The Core Cycle
```
Set a Goal
     |
     v
Write an Initial Prompt
     |
     v
Eval the Prompt
     |
     v
Apply a Prompt Engineering Technique  <---+
     |                                    |
     v                                    |
Re-eval to Verify Better Performance -----+
                   (repeat until satisfied)
```

Make **one change at a time** — so you know exactly what caused the improvement.

---

## Step-by-Step Breakdown

### 1. Set a Goal
Define what the prompt must accomplish and what "good output" looks like.

**Example goal:** Generate a one-day meal plan for an athlete given their height,
weight, goal, and dietary restrictions.

### 2. Write an Initial Prompt
Start with a simple, naive prompt. This becomes your **baseline score**.
```python
def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)
```

> A score of ~2–3 out of 10 on a first attempt is normal. The baseline exists to
> measure improvement, not to be good.

### 3. Eval the Prompt

Generate a dataset and run evaluation with specific criteria:
```python
evaluator = PromptEvaluator(max_concurrent_tasks=3)  # start low to avoid rate limits

dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    output_file="dataset.json",
    num_cases=3                  # keep low during development
)

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""
)
```

### 4. Apply a Prompt Engineering Technique
Use the eval report to identify weaknesses, then apply one technique to address them.
Techniques include: clearer instructions, XML tags, multishot examples, role assignment, etc.

### 5. Re-eval to Verify Better Performance
Re-run the full eval against the same dataset. Compare the new average score to
the baseline. If it improved, keep the change. If not, revert and try a different technique.

---

## Practical Tips

| Tip | Detail |
|---|---|
| Start with low concurrency | Use `max_concurrent_tasks=3` initially to avoid rate limit errors |
| Keep dataset small during dev | 2–3 cases is enough for fast iteration |
| Change one thing at a time | Makes it clear what caused score changes |
| Use the HTML eval report | Shows per-test reasoning — tells you *why* a test case scored low |
| Don't chase perfection early | Focus on consistent upward trend in scores |

---

## Key Takeaway

> Prompt engineering is not guesswork — it is a **systematic, measurement-driven cycle**.
> Each technique you apply should move the average eval score upward.
> The eval pipeline is what separates engineering from trial and error.

# Prompt Engineering Technique 1 — Be Clear and Direct

---

## Why the First Line Matters

The first line of a prompt sets the stage for everything that follows.
A weak opening leads to vague, unfocused responses. A strong opening immediately
anchors the model to the right task.

---

## Two Principles

### "Clear" — Remove Ambiguity

| Rule | Detail |
|---|---|
| Use simple language | No vague or roundabout phrasing |
| State what you want explicitly | Don't make Claude infer the task |
| Lead with a statement of the task | The task should be the very first thing |

**Instead of:**
> "I need to know about those things people put on their roofs that use sun —
> those solar panel things, I think they're called"

**Use:**
> "Write three paragraphs about how solar panels work"

---

### "Direct" — Use Instructions, Not Questions

| Rule | Detail |
|---|---|
| Use instructions, not questions | Commands produce more focused output |
| Start with action verbs | "Write", "Create", "Generate", "Identify", "List" |

**Instead of:**
> "I was reading about renewable energy and geothermal energy sounds neat.
> What countries use it?"

**Use:**
> "Identify three countries that use geothermal energy. Include generation stats for each."

---

## Applied Example — Meal Plan Prompt

**Weak v1:**
```
What should this person eat?
```

**Improved v2 (clear and direct):**
```
Generate a one-day meal plan for an athlete that meets their dietary restrictions.
```

The improved version tells Claude immediately:
- What action to take → `Generate`
- What to create → a one-day meal plan
- Who it is for → an athlete
- Key constraint → must meet dietary restrictions

**Eval score impact: 2.32 → 3.92**

---

## Key Takeaway

> Claude performs best when treated as a capable assistant who needs **clear direction**,
> not someone who has to guess what you want.
> Start with a direct action verb. State the task explicitly. Specify constraints upfront.

# Prompt Engineering Technique 2 — Be Specific

---

## The Problem with Vague Prompts

A vague prompt like "write a short story about a character who discovers a hidden talent"
leaves too many decisions to Claude:
- Length: 200 words or 2,000?
- Characters: one or five?
- Direction: countless possibilities

Specificity gives Claude a **clear target** — improving both consistency and quality.

---

## Two Types of Specificity

### Type 1 — Output Quality Guidelines

A list of qualities the output **must have**. Controls:
- Length and format
- Specific elements to include
- Tone, structure, style

**Example — meal plan prompt guidelines:**
```
Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
```

**Eval score impact: 3.92 → 7.86** (more than doubled from adding guidelines alone)

---

### Type 2 — Process Steps

Step-by-step instructions for **how Claude should think through** the problem.
Useful when you want systematic reasoning before a final answer.

**Example — story writing:**
```
1. Brainstorm three talents that would create dramatic tension
2. Pick the most interesting one
3. Outline a pivotal scene that reveals the talent
4. Brainstorm supporting character types that could increase impact
5. Write the story
```

**Example — sales performance analysis:**
```
1. Examine market metrics and industry changes
2. Review individual performance data
3. Check for organizational changes
4. Analyse customer feedback
5. Summarise root causes and recommend actions
```

This prevents Claude from latching onto a single cause and ensures all angles are covered.

---

## When to Use Each

| Situation | Use |
|---|---|
| Almost every prompt | Output quality guidelines |
| Complex troubleshooting | Process steps |
| Decision-making scenarios | Process steps |
| Critical thinking tasks | Process steps |
| Consistent, formatted output | Output quality guidelines |
| Multi-angle analysis needed | Both combined |

---

## Combining Both (Professional Pattern)
```
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts
3. Specify when to eat each meal
4. Use only foods that fit the athlete's restrictions
5. List all portion sizes in grams

Steps:
1. Calculate estimated daily caloric needs based on athlete stats
2. Determine macro split appropriate for the stated goal
3. Assign foods to meals respecting the dietary restrictions
4. Verify totals add up correctly before responding
```

---

## Key Takeaway

> Output guidelines control **what** Claude produces.  
> Process steps control **how** Claude thinks before producing it.  
> Used together, they give you consistent output and confident reasoning.

# Prompt Engineering Technique 3 — XML Tags for Structure

---

## The Problem

When a prompt contains large amounts of data mixed with instructions, Claude can struggle
to distinguish between:
- The instructions you are giving
- The data you want processed

This is especially problematic with 20+ lines of records, code, or multi-variable inputs.

---

## The Solution: XML Tags as Delimiters

Wrap distinct sections of content in descriptive XML tags to create **clear boundaries**.

**Without tags — unclear:**
```
Debug this code using the documentation below.

def calculate(x, y): return x / y

Returns the quotient of two numbers. Raises ZeroDivisionError if y is 0.
Use try/except to handle division errors gracefully.
```

**With tags — clear:**
```
Debug this code using the documentation below.

<my_code>
def calculate(x, y):
    return x / y
</my_code>

<docs>
Returns the quotient of two numbers. Raises ZeroDivisionError if y is 0.
Use try/except to handle division errors gracefully.
</docs>
```

---

## Choosing Good Tag Names

You don't need to use official XML — invent descriptive names that match your content.

| Less useful | More useful |
|---|---|
| `<data>` | `<sales_records>` |
| `<input>` | `<athlete_information>` |
| `<text>` | `<customer_feedback>` |
| `<code>` | `<my_code>` |

The more descriptive the tag name, the better Claude understands the **purpose** of
each section.

---

## Practical Example — Meal Plan Prompt
```
<athlete_information>
- Height: 6'2"
- Weight: 180 lbs
- Goal: Build muscle
- Dietary restrictions: Vegetarian
</athlete_information>

Generate a meal plan based on the athlete information above.
```

Claude now clearly knows that all four data points are related athlete attributes
to use together — not separate instructions.

---

## When to Use XML Tags

| Situation | Use tags? |
|---|---|
| Large amounts of context or data | Yes — always |
| Mixing code and documentation | Yes — always |
| Multiple interpolated variables | Yes — strongly recommended |
| Complex prompts with many sections | Yes |
| Short, simple single-topic prompts | Optional, still helpful |

---

## Key Takeaway

> XML tags are **delimiters** — they tell Claude where one type of content ends
> and another begins.  
> They become increasingly valuable as prompts grow longer and more complex.  
> Always prefer descriptive tag names over generic ones.

# Prompt Engineering Technique 4 — Examples (One-Shot & Multi-Shot)

---

## What is Shot Prompting?

Providing example input/output pairs inside your prompt to **show** Claude what good
output looks like, rather than only describing it in words.

| Term | Meaning |
|---|---|
| **Zero-shot** | No examples — Claude infers from instructions alone |
| **One-shot** | One example input/output pair |
| **Multi-shot** | Multiple examples covering different scenarios |

---

## Why Examples Work

Some requirements are hard to describe in words but easy to demonstrate.

**Problem — sarcasm in sentiment analysis:**
```
Classify this tweet as Positive or Negative:
"Yeah, sure, that was the best movie I've seen since Plan 9 from Outer Space"
```

Without context, Claude may classify this as Positive. With an example of sarcasm,
it learns to handle it correctly.

**With a multi-shot example:**
```
Classify tweets as Positive or Negative. Watch carefully for sarcasm.

<example>
<sample_input>Oh yeah, I really needed a flight delay tonight! Excellent!</sample_input>
<ideal_output>Negative</ideal_output>
Note: This is sarcastic — the speaker is frustrated, not pleased.
</example>

<example>
<sample_input>Great game tonight!</sample_input>
<ideal_output>Positive</ideal_output>
</example>
```

---

## When to Use Examples

- Handling **corner cases and edge scenarios**
- Defining **complex or exact output formats** (specific JSON structures, tables)
- Showing **style, tone, or formatting** that is hard to describe
- Demonstrating how to handle **ambiguous inputs**

---

## Structure Your Examples with XML Tags

Always wrap examples clearly so Claude knows what is input vs. expected output:
```
<example>
<sample_input>
[input here]
</sample_input>
<ideal_output>
[output here]
</ideal_output>
This example is good because [explain why].
</example>
```

The explanation after the output helps Claude understand the **reasoning**, not just
the format.

---

## Finding Good Examples from Your Eval Results

When running evaluations, look for your **highest-scoring outputs** (score = 10):
```python
best_results = [r for r in results if r["score"] == 10]

# Use best_results[0]["test_case"] as sample_input
# Use best_results[0]["output"]    as ideal_output
```

Real outputs that scored perfectly make the most reliable examples because they
represent what "good" actually looks like for your specific task.

---

## Best Practices

| Rule | Detail |
|---|---|
| Always use XML tags | Makes input vs output unambiguous |
| Be explicit | Label things: "Here is an example input with an ideal response" |
| Cover failure cases | Use examples that address your most common eval failures |
| Explain why the output is good | Context helps Claude generalise, not just copy |
| Keep examples task-relevant | Off-topic examples can confuse more than help |

---

## Key Takeaway

> Examples **show** rather than **tell**.  
> They are especially powerful for subtle requirements — tone, sarcasm, format nuance —
> that are difficult to fully capture in written instructions.  
> Use your best eval outputs as examples to anchor Claude to your quality bar.